# A second format: ASEG-GDF

## GSPy workshop, part 2

Part 1 built a GS file from CSV files, where the header gives you column
names and nothing else. This notebook walks the same path for a Tempest AEM
survey stored in **ASEG-GDF2**, a format that carries its own column
definitions.

The interesting question is how much less work you have to do.

| | Section | Time |
|---|---|---|
| 1 | What is in the box? | 5 min |
| 2 | A template that fills itself in | 10 min |
| 3 | Survey and raw data | 8 min |
| 4 | Models derived from data | 7 min |
| 5 | A raster map | 5 min |
| 6 | Save, reopen, inspect | 5 min |

Roughly 40 minutes. Cells marked **EXERCISE** need completing; solutions are
in `solutions/`.

*Source data: Minsley, B.J., James, S.R., Bedrosian, P.A., Pace, M.D.,
Hoogenboom, B.E., and Burton, B.L., 2021, Airborne electromagnetic,
magnetic, and radiometric survey of the Mississippi Alluvial Plain,
November 2019 - March 2020: U.S. Geological Survey data release,
https://doi.org/10.5066/P9E44CTQ*

In [ ]:
import warnings
from os.path import join
from pathlib import Path

import matplotlib.pyplot as plt

import gspy
from gspy import Dataset, Survey

warnings.filterwarnings('ignore')

DATA = 'data/tempest'
PREPARED = 'prepared_metadata/tempest'
SCRATCH = 'my_metadata'

Path(SCRATCH).mkdir(exist_ok=True)
assert Path(DATA).is_dir(), 'Run Jupyter from the workshop root folder'
print('gspy', gspy.__version__)

---
---
## 1. What is in the box?

In [ ]:
for path in sorted(Path(DATA).iterdir()):
    print(f'{path.name:<28} {path.stat().st_size / 1e6:>6.1f} MB')

An ASEG-GDF dataset comes in pairs: a `.dat` file holding whitespace
delimited numbers with **no header at all**, and a `.dfn` definition file
describing the columns. Look at the first few lines of each.

In [ ]:
print('--- Tempest.dfn ---')
print('\n'.join(Path(DATA, 'Tempest.dfn').read_text().splitlines()[:8]))
print()
print('--- Tempest.dat ---')
with open(join(DATA, 'Tempest.dat')) as f:
    for _ in range(2):
        print(f.readline()[:140], '...')

Each `DEFN` line names a column and describes it:

```
DEFN 15 ST=RECD,RT=;GPS_Elevation:f8.2:UNIT=m:NULL=-999.99,NAME=Final GPS Elevation (Ortho)
```

That single line carries the column name, its Fortran format, its **units**,
its **no-data value** and a **descriptive name**. Compare that with a CSV
header, which gave you the word `Alt` and left you guessing.

---
### Exercise 1.1 - discussion, no code

Which of the questions that stumped us in part 1 section 1 could you now
answer from the DFN alone? Which ones still could not be answered?

*Your answer:*

---
---
## 2. A template that fills itself in

Same call as part 1, pointed at the `.dat` file. GSPy finds the matching
`.dfn` on its own.

In [ ]:
template = Dataset.metadata_template(join(DATA, 'Tempest.dat'))
template.dump(join(SCRATCH, 'tempest_raw_template.yml'))

print(f"{len(template['variables'])} variables found\n")
for name in ('Line', 'GPS_Elevation', 'DTM', 'EMX_HPRG'):
    print(f'{name}:')
    for key, value in template['variables'][name].items():
        print(f'    {key}: {value}')
    print()

Compare that with part 1, where all four fields came back `not_defined`.
Here `long_name`, `units` and `missing_value` are already populated,
straight out of the DFN, for 61 variables.

`EMX_HPRG` shows the same `dimensions: ['index', '??']` as `HM_X` did - the
DFN says there are 15 windows, but not what time each window corresponds to.
Gate times are a property of the instrument, so they still have to come from
a system definition. Some things a file format cannot tell you.

---
### Exercise 2.1

Find out what is still missing. Count how many variables have at least one
`not_defined` field, and show which fields those are.

*Hint: iterate over `template['variables'].items()` and look for values
equal to `'not_defined'`.*

In [ ]:
# --------------------------------------------------------------------
# EXERCISE - complete the lines marked with ...
# --------------------------------------------------------------------
incomplete = {}
for name, attrs in template['variables'].items():
    missing = ...
    assert isinstance(missing, list), 'missing should be a list of attribute names'
    if missing:
        incomplete[name] = missing

print(f'{len(incomplete)} of {len(template["variables"])} variables have gaps\n')
for name, missing in list(incomplete.items())[:10]:
    print(f'{name:<20} {missing}')

Mostly `units`, on variables that genuinely have none - line numbers, flight
numbers, dates - plus the handful the DFN left blank. The DFN got you most
of the way; the rest is judgement.

Notice also what the DFN could *not* provide: which columns are the
coordinates. `Easting_Albers` and `Northing_Albers` are obvious to you and
invisible to software.

In [ ]:
print('coordinates GSPy still needs:')
for axis, value in template['coordinates'].items():
    print(f'  {axis}: {value}')

---
---
## 3. Survey and raw data

From here the workflow is identical to part 1, so we move at pace with the
prepared metadata.

In [ ]:
survey = Survey.from_dict(join(PREPARED, 'Tempest_survey_md.yml'))
print(survey.gs.tree)

survey

A survey with no children, same as part 1. The system definition is placed
differently this time, though: instead of a standalone `skytem_system.yml`
handed to `system=`, the Tempest system is written *inside*
`Tempest_data_md.yml`, and a container lifts out any top level key whose name
contains `system`.

Both are valid. Which you want depends on whether the instrument description
is shared: a standalone file when several datasets cite one instrument, an
inline block when it describes only this dataset.

---
### Exercise 3.1

Create a `data` container and attach `Tempest.dat` using
`Tempest_data_md.yml`. No `system=` argument is needed, because the system
comes in with the metadata file.

In [ ]:
# --------------------------------------------------------------------
# EXERCISE - complete the lines marked with ...
# --------------------------------------------------------------------
data_container = survey.gs.add_container(..., content='raw data')

raw_data = data_container.gs.add(
    key=...,
    data=...,
    metadata_file=...)

print(dict(raw_data.sizes))
print(survey.gs.tree)

raw_data

`gate_times: 15` resolved the `??`, and `tempest_system` now hangs below the
raw data as its own group.

---
---
## 4. Models derived from data

The inverted models were produced from that raw data with that same
instrument, and `system=raw_data.tempest_system` records it: the models
cite the *same* system definition rather than describing it a second time.

---
### Exercise 4.1

Attach `Tempest_model.dat` with `Tempest_model_md.yml` to a new `models`
container, passing the system from the raw data.

In [ ]:
# --------------------------------------------------------------------
# EXERCISE - complete the lines marked with ...
# --------------------------------------------------------------------
model_container = survey.gs.add_container(..., content='inverted 1-D electrical resistivity models')

models = model_container.gs.add(
    key=...,
    data=...,
    metadata_file=...,
    system=...)

print(dict(models.sizes))

models

`layer_depth: 30` alongside `gate_times: 15` - the models carry both their
own layer geometry and the gate times of the data they were fitted to.

---
---
## 5. A raster map

---
### Exercise 5.1

Add the contractor's total magnetic intensity grid to a `derived_maps`
container, using `Tempest_raster_md.yml`. Remember rasters take no
`data`.

In [ ]:
# --------------------------------------------------------------------
# EXERCISE - complete the lines marked with ...
# --------------------------------------------------------------------
map_container = survey.gs.add_container(..., content='derived maps')

maps = map_container.gs.add(
    key=...,
    metadata_file=...)

print(dict(maps.sizes))

maps

---
---
## 6. Save, reopen, inspect

In [ ]:
out_file = join(SCRATCH, 'tempest_workshop.nc')
survey.gs.to_netcdf(out_file)

reopened = gspy.open_datatree(out_file)['survey']
print(reopened.gs.tree)

---
### Exercise 6.1

The same four questions as part 1 section 6, now on this file. Pick any
variable in the raw data and report its long name, units and no-data value -
then check them against the `DEFN` line for that column in `Tempest.dfn`.
The whole point is that they agree.

In [ ]:
# --------------------------------------------------------------------
# EXERCISE - complete the lines marked with ...
# --------------------------------------------------------------------
raw = reopened['data']['raw_data']

variable = ...      # note: GSPy lower-cases variable names on the way in
for key, value in ...:
    print(f'{key:<16} {value}')

print()
for line in Path(DATA, 'Tempest.dfn').read_text().splitlines():
    if ... in line:
        print(line.strip())

---
### Exercise 6.2

Two plots to finish: a scatter of the raw data coloured by transmitter
height, and the magnetic grid.

*Hint: `.gs.scatter(x='x', hue=...)` for the tabular data. The grid variable
is `magnetic_tmi`.*

In [ ]:
# --------------------------------------------------------------------
# EXERCISE - complete the lines marked with ...
# --------------------------------------------------------------------
plt.figure(figsize=(7, 6))
reopened['data']['raw_data'].gs.scatter(x='x', hue=..., cmap='jet')
plt.tight_layout()

plt.figure(figsize=(7, 6))
reopened[...][...].plot(cmap='jet', robust=True)
plt.tight_layout()

---
---
## Taking stock

Same eight-step workflow, two very different input formats. What changed was
only how much of the variable metadata came for free:

| | CSV (part 1) | ASEG-GDF (part 2) |
|---|---|---|
| variable names | from the header | from the DFN |
| long names | you write them | from the DFN |
| units | you write them | from the DFN |
| no-data values | you write them | from the DFN |
| which columns are coordinates | you say so | you say so |
| gate times, instrument geometry | a system definition | a system definition |

The two rows at the bottom never come for free, in any format. They are
knowledge about the survey, not about the file, and writing them down once
in a system definition is the most valuable thing in this workshop.

- GSPy documentation: https://doi-usgs.github.io/gspy